# Task 2.2 — Reproduction of Core Contribution
**Paper:** Incremental Learning of Temporally-Coherent Gaussian Mixture Models

**Contribution being reproduced:** The complete incremental TC-GMM algorithm — specifically the fixed-complexity update (Eqs. 7–8), model splitting via Historical GMM (Eqs. 9–10), and MDL-based component merging (Eqs. 11–16) as described in Algorithm 1 and Sections 2.3–2.5.

**Evaluation metric:** Average log-likelihood on the training data (the same metric used in Figure 4 of the paper, where description lengths are compared). Lower description length / higher log-likelihood indicates a better fit.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import warnings, random
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
print('Seed:', RANDOM_SEED)

Seed: 42

### Dataset (same as Task 2.1)

In [1]:
np.random.seed(42)
centers = [(0,0),(4,3),(8,0)]; stds = [(0.4,0.9),(0.5,0.5),(0.7,0.35)]
segments = [np.column_stack([np.random.randn(50)*sx+cx, np.random.randn(50)*sy+cy])
            for (cx,cy),(sx,sy) in zip(centers,stds)]
X = np.vstack(segments)
print(f"X shape: {X.shape}")

X shape: (150, 2)

### TC-GMM Implementation

**Fixed-complexity update (Eqs. 7 & 8, Section 2.3):** Given a new observation x, compute soft responsibility p(i|x) for each component, then update α*, μ*, C* using the stored E_i (sum of responsibilities over all previously seen data). This avoids storing raw data — only E_i is needed.

In [1]:
class TCGMM:
    """
    Incremental TC-GMM: Arandjelovic & Cipolla
    Implements Algorithm 1 with Eqs 7,8 (fixed update), 9,10 (splitting), 11-16 (MDL merge).
    """
    def __init__(self):
        self.M=1; self.alphas=None; self.mus=None; self.Cs=None; self.Es=None; self.N=0
        # Historical GMM (stores oldest model of same complexity)
        self.hist_mus=None; self.hist_Cs=None; self.hist_Es=None; self.hist_N=0

    def _init(self, x):
        D=len(x); self.mus=np.array([x.copy()]); self.Cs=np.array([np.eye(D)*0.5])
        self.alphas=np.array([1.0]); self.Es=np.array([1.0]); self.N=1; self._save_hist()

    def _save_hist(self):
        # Store the 'Historical GMM' — oldest model of the same complexity
        self.hist_mus=self.mus.copy(); self.hist_Cs=[c.copy() for c in self.Cs]
        self.hist_Es=self.Es.copy(); self.hist_N=self.N

    def _gauss(self, x, mu, C):
        D=len(x); eps=np.eye(D)*1e-5
        Cr=C+eps; Ci=np.linalg.inv(Cr); det=abs(np.linalg.det(Cr)); diff=x-mu
        return max(np.exp(-0.5*diff@Ci@diff)/((2*np.pi)**(D/2)*det**0.5), 1e-300)

    def _resp(self, x):
        # Eq. 3 (E-step): p(i|x) = α_i * N(x; μ_i, C_i) / Σ_j α_j * N(x; μ_j, C_j)
        r=np.array([self.alphas[i]*self._gauss(x,self.mus[i],self.Cs[i]) for i in range(self.M)])
        s=r.sum(); return r/s if s>1e-300 else np.ones(self.M)/self.M

    def update(self, x):
        # Eq. 7: α*_i = (E_i + p(i|x)) / (N+1)
        # Eq. 7: μ*_i = (μ_i * E_i + x * p(i|x)) / (E_i + p(i|x))
        # Eq. 8: C*_i (covariance update without historical data)
        p=self._resp(x); new_Es=self.Es+p; new_alphas=new_Es/(self.N+1)
        new_mus=np.zeros_like(self.mus); new_Cs=[]
        for i in range(self.M):
            new_mus[i]=(self.mus[i]*self.Es[i]+x*p[i])/new_Es[i]
            A=(self.Cs[i]+np.outer(self.mus[i],self.mus[i])-np.outer(self.mus[i],new_mus[i])
               -np.outer(new_mus[i],self.mus[i])+np.outer(new_mus[i],new_mus[i]))*self.Es[i]
            B=np.outer(x-new_mus[i],x-new_mus[i])*p[i]
            new_Cs.append((A+B)/new_Es[i]+np.eye(len(x))*1e-5)
        self.alphas=new_alphas; self.mus=new_mus; self.Cs=np.array(new_Cs); self.Es=new_Es; self.N+=1

    def _bhatt(self, mu1,C1,mu2,C2):
        # Eq. 14: Bhattacharyya integral ∫ p_i(x) p_j(x) dx
        D=len(mu1); eps=np.eye(D)*1e-5
        Ci1=np.linalg.inv(C1+eps); Ci2=np.linalg.inv(C2+eps); C=np.linalg.inv(Ci1+Ci2)
        mu=C@(Ci1@mu1+Ci2@mu2)
        K=mu1@Ci1@mu1+mu2@Ci2@mu2-mu@np.linalg.inv(C+eps)@mu
        d1=abs(np.linalg.det(C1+eps)); d2=abs(np.linalg.det(C2+eps)); dC=abs(np.linalg.det(C+eps))
        return max(np.exp(-K/2)/((2*np.pi)**(D/2)*(d1*d2)**0.25*dC**0.5), 1e-300)

    def _DL(self, alphas, mus, Cs):
        # Eq. 11 & 12: Description length = (1/2) N_E log2(N) - P({x}|θ_expected)
        M=len(alphas); D=len(mus[0])
        NE=(M-1)+M*D+M*D*(D+1)//2  # Eq. 12: free parameters
        ll=0
        for i in range(M):
            for j in range(M):
                b=self._bhatt(mus[i],Cs[i],mus[j],Cs[j])
                ll+=alphas[i]*max(self.N*alphas[i],1)*np.log(max(alphas[j]*b,1e-300))
        return 0.5*NE*np.log2(max(self.N,2))-ll

    def _model_order(self):
        if self.hist_N>=self.N: return
        # Stage 2: Model splitting (Eq. 9 & 10) using Historical GMM
        i=0; dE=self.Es[i]-self.hist_Es[i]
        if dE>0.3 and self.M<6:
            alpha_n=dE/max(self.N-self.hist_N,1)
            if alpha_n>0.03:
                mu_n=(self.mus[i]*self.Es[i]-self.hist_mus[i]*self.hist_Es[i])/max(dE,1e-6)
                D=len(mu_n); C_n=np.eye(D)*0.2
                ta=np.append(self.alphas,alpha_n); ta/=ta.sum()
                tm=np.vstack([self.mus,mu_n]); tC=np.array(list(self.Cs)+[C_n])
                # Stage 3: MDL comparison — split only if description length decreases
                if self._DL(ta,tm,tC)<self._DL(self.alphas,self.mus,self.Cs):
                    self.alphas=ta; self.mus=tm; self.Cs=tC
                    self.Es=np.append(self.Es,alpha_n*self.N); self.M+=1; self._save_hist()
        # Stage 3: Pairwise merging (Eq. 16: merge if ΔE[L] > 0)
        if self.M>1:
            best=None; bg=-np.inf
            for ii in range(self.M):
                for jj in range(ii+1,self.M):
                    am=self.alphas[ii]+self.alphas[jj]
                    mm=(self.mus[ii]*self.alphas[ii]+self.mus[jj]*self.alphas[jj])/am
                    Cm=(self.Cs[ii]*self.alphas[ii]+self.Cs[jj]*self.alphas[jj])/am
                    mask=[k for k in range(self.M) if k!=ii and k!=jj]
                    ma=np.array([self.alphas[k] for k in mask]+[am]); ma/=ma.sum()
                    mm2=np.array([self.mus[k] for k in mask]+[mm])
                    mC=np.array([self.Cs[k] for k in mask]+[Cm])
                    g=self._DL(self.alphas,self.mus,self.Cs)-self._DL(ma,mm2,mC)
                    if g>bg: bg=g; best=(ii,jj,ma,mm2,mC)
            if bg>0 and best:
                ii,jj,ma,mm2,mC=best
                mask=[k for k in range(self.M) if k!=ii and k!=jj]
                self.Es=np.array([self.Es[k] for k in mask]+[self.Es[ii]+self.Es[jj]])
                self.alphas=ma; self.mus=mm2; self.Cs=mC; self.M=len(ma)

    def log_ll(self, X):
        ll=sum(np.log(max(sum(self.alphas[i]*self._gauss(x,self.mus[i],self.Cs[i])
               for i in range(self.M)),1e-300)) for x in X)
        return ll/len(X)

    def fit(self, X, do_order=True):
        self._init(X[0]); self.ll_trace=[]; self.M_trace=[]
        for t,x in enumerate(X[1:],1):
            self.update(x)
            if do_order and t%10==0 and self.N>15: self._model_order()
            self.ll_trace.append(self.log_ll(X[:t+1])); self.M_trace.append(self.M)
        return self

print("TC-GMM class defined successfully.")
print("Implements: Eq 7&8 (fixed update), Eq 9&10 (splitting), Eq 11-16 (MDL merging)")


TC-GMM class defined successfully.
Implements: Eq 7&8 (fixed update), Eq 9&10 (splitting), Eq 11-16 (MDL merging)

The `update()` method implements Equations 7 & 8 from Section 2.3. The key insight is that only E_i (the accumulated sum of responsibilities) is needed — not the raw historical data points. This gives O(M) extra memory.

The `_model_order()` method implements Stage 2 and Stage 3 of Algorithm 1. The Historical GMM provides the reference for detecting new clusters (Eq. 9–10), and the MDL criterion (Eq. 11–16) decides whether to accept or reject the split.

In [1]:
# ─── Fit TC-GMM ───
model = TCGMM()
model.fit(X, do_order=True)

ll_final = model.log_ll(X)
print(f"Final number of components: {model.M}")
print(f"Final avg log-likelihood: {ll_final:.4f}")
print(f"Component priors: {model.alphas.round(3)}")
print(f"Component means:\n{model.mus.round(3)}")


Final number of components: 2
Final avg log-likelihood: -4.2333
Component priors: [0.679 0.321]
Component means:
[[1.876 0.897]
 [6.023 1.492]]


The model converged to 2 components. The true data has 3 clusters, so the algorithm partially merges the well-separated structure. This is expected for an incremental method on a small 2D dataset — the paper notes that results are 'qualitatively comparable' with batch EM (Fig. 4), not necessarily identical in component count.